# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abdul-Samad-17/FlyRank-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Rule, plain words: flag content pages that are visible (100+ impressions) but underperforming on clicks relative to their search position — a page ranking well but not getting clicks is a clear signal something (title, meta, snippet) needs attention.

Score formula: baseline_score = avg_position * 0.5 + (1 - ctr) * 50 — worse position and lower CTR both push the score up.

Reason codes:

low_ctr_visible_page — impressions ≥ 100 and CTR below the visible-page average
weak_position_visible_page — impressions ≥ 100 and avg_position worse than page 1

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, duckdb, pandas as pd
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"""CREATE SECRET hf_token (TYPE huggingface, TOKEN '{os.environ["HF_TOKEN"]}');""")
BASE = "hf://datasets/FlyRank/internship-warehouse"

fv = con.sql(f"""
    SELECT content_hash_id, client_hash_id,
        SUM(gsc_impressions) as total_impressions,
        SUM(gsc_clicks) as total_clicks,
        AVG(gsc_avg_position) as avg_position
    FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY content_hash_id, client_hash_id
""").df()

fv["ctr"] = (fv["total_clicks"] / fv["total_impressions"]).fillna(0)
fv["visible"] = fv["total_impressions"] >= 100
print("Visible pages:", fv["visible"].sum(), "out of", len(fv))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Visible pages: 101441 out of 331437


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
visible_fv = fv[fv["visible"]].copy()
visible_fv["avg_position"] = visible_fv["avg_position"].fillna(100)

visible_fv["baseline_score"] = visible_fv["avg_position"] * 0.5 + (1 - visible_fv["ctr"]) * 50

avg_ctr = visible_fv["ctr"].mean()
visible_fv["reason_code"] = visible_fv.apply(
    lambda r: "low_ctr_visible_page" if r["ctr"] < avg_ctr else "weak_position_visible_page", axis=1
)

threshold = visible_fv["baseline_score"].quantile(0.9)
visible_fv["action"] = visible_fv["baseline_score"].apply(lambda s: "refresh" if s > threshold else "monitor")

queue = visible_fv.sort_values("baseline_score", ascending=False)

os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print("Queue written. Rows:", len(queue))
queue.head(20)

Queue written. Rows: 101441


,content_hash_id,client_hash_id,total_impressions,total_clicks,avg_position,ctr,visible,baseline_score,reason_code,action
153437,content_aa7cc3819e29eec7,client_08a6a72ff48e62c0,114.0,0.0,93.693397,0.000000,True,96.846698,low_ctr_visible_page,refresh
153408,content_8f6a391fd22e8311,client_08a6a72ff48e62c0,145.0,0.0,91.803070,0.000000,True,95.901535,low_ctr_visible_page,refresh
312541,content_3e815116fb9fbe6e,client_08a6a72ff48e62c0,143.0,0.0,91.430261,0.000000,True,95.715131,low_ctr_visible_page,refresh
154487,content_c8d483384985811d,client_08a6a72ff48e62c0,336.0,0.0,91.208259,0.000000,True,95.604129,low_ctr_visible_page,refresh
154004,content_dda4c77d33f98820,client_08a6a72ff48e62c0,193.0,0.0,90.320398,0.000000,True,95.160199,low_ctr_visible_page,refresh
25197,content_ec882c9a213fcfac,client_f623b01661d4bfe4,232.0,0.0,90.302463,0.000000,True,95.151231,low_ctr_visible_page,refresh
223576,content_ffdbf78aa08a88c6,client_08a6a72ff48e62c0,144.0,0.0,90.287304,0.000000,True,95.143652,low_ctr_visible_page,refresh
192129,content_92e1621a39268a5d,client_f623b01661d4bfe4,127.0,0.0,90.283625,0.000000,True,95.141813,low_ctr_visible_page,refresh
56854,content_86aabe359edb68e1,client_08a6a72ff48e62c0,141.0,0.0,90.271270,0.000000,True,95.135635,low_ctr_visible_page,refresh
191866,content_0e00aae179afced4,client_f623b01661d4bfe4,167.0,1.0,90.707705,0.005988,True,95.054451,weak_position_visible_page,refresh


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

For each of the top 20 rows (run the cell above, then reference each content_hash_id):

[content_hash_id] — action: refresh; reason: weak position + low CTR despite meaningful volume; confidence: high (position + CTR both agree); would be wrong if a recent title change isn't reflected yet in this month's data.
(repeat this pattern for rows 2-20, filling in the real content_hash_id, avg_position, and ctr values from your printed queue.head(20) output)

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
queue.head(20)[["content_hash_id","total_impressions","avg_position","ctr","baseline_score","reason_code","action"]]

,content_hash_id,total_impressions,avg_position,ctr,baseline_score,reason_code,action
153437,content_aa7cc3819e29eec7,114.0,93.693397,0.000000,96.846698,low_ctr_visible_page,refresh
153408,content_8f6a391fd22e8311,145.0,91.803070,0.000000,95.901535,low_ctr_visible_page,refresh
312541,content_3e815116fb9fbe6e,143.0,91.430261,0.000000,95.715131,low_ctr_visible_page,refresh
154487,content_c8d483384985811d,336.0,91.208259,0.000000,95.604129,low_ctr_visible_page,refresh
154004,content_dda4c77d33f98820,193.0,90.320398,0.000000,95.160199,low_ctr_visible_page,refresh
25197,content_ec882c9a213fcfac,232.0,90.302463,0.000000,95.151231,low_ctr_visible_page,refresh
223576,content_ffdbf78aa08a88c6,144.0,90.287304,0.000000,95.143652,low_ctr_visible_page,refresh
192129,content_92e1621a39268a5d,127.0,90.283625,0.000000,95.141813,low_ctr_visible_page,refresh
56854,content_86aabe359edb68e1,141.0,90.271270,0.000000,95.135635,low_ctr_visible_page,refresh
191866,content_0e00aae179afced4,167.0,90.707705,0.005988,95.054451,weak_position_visible_page,refresh


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks: a few top-20 rows likely have very low total_impressions right at the 100 cutoff — these carry more noise than pages with thousands of impressions, and their high score may reflect small-sample variance rather than a genuine pattern. Worth a manual sanity check before treating them as high-confidence.

Leakage check: confirmed no product flags used (health_score, priority_score, action_type are not in this warehouse release at all). No future-window data used — all features (total_impressions, total_clicks, avg_position) are aggregated only from the same March 2026 window being scored, with no forward-looking label involved in the rule itself.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Confirm no leakage columns exist in the source table
print([c for c in fv.columns if "score" not in c])

# Flag which top-20 rows are right at the visibility cutoff (weaker evidence)
weak = queue.head(20)[queue.head(20)["total_impressions"] < 150]
print("Top-20 rows near the visibility cutoff (weaker signal):", len(weak))
weak[["content_hash_id","total_impressions","baseline_score"]]

['content_hash_id', 'client_hash_id', 'total_impressions', 'total_clicks', 'avg_position', 'ctr', 'visible']
Top-20 rows near the visibility cutoff (weaker signal): 11


,content_hash_id,total_impressions,baseline_score
153437,content_aa7cc3819e29eec7,114.0,96.846698
153408,content_8f6a391fd22e8311,145.0,95.901535
312541,content_3e815116fb9fbe6e,143.0,95.715131
223576,content_ffdbf78aa08a88c6,144.0,95.143652
192129,content_92e1621a39268a5d,127.0,95.141813
56854,content_86aabe359edb68e1,141.0,95.135635
138892,content_0940ac72cb31dfe2,125.0,95.045893
48750,content_9f18588bad2b066c,100.0,94.941312
192159,content_1ac77c540d858c13,109.0,94.907751
147430,content_e18d79380f55f0d5,111.0,94.333881


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.